In [1]:
# USAGE
# python detect_mask_image.py --image examples/example_01.png

# import the necessary packages
import os
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import load_model
import numpy as np
import argparse
import cv2

import ipywidgets as widgets
from IPython.display import display

In [ ]:


path = os.path.join("mask_detector.h5")

print(path)

TypeError: join() missing 1 required positional argument: 'a'

In [8]:
#load our serialized face detector model from disk
print("[INFO] loading face detector model...")
prototxtPath = os.path.join("face_detector", "deploy.prototxt")
weightsPath = os.path.join("face_detector", "res10_300x300_ssd_iter_140000.caffemodel")
net = cv2.dnn.readNet(prototxtPath, weightsPath)

# load the face mask detector model from disk
print("[INFO] loading face mask detector model...")
modelPath = "mask_detector.h5"
model = load_model(modelPath, compile=False)


[INFO] loading face detector model...
[INFO] loading face mask detector model...


In [9]:
# load the input image from disk, clone it, and grab the image spatial
# dimensions. Change this to any JPG or PNG you want to test.
# imagePath = os.path.join("dataset", "with_mask", "with_mask388.jpg")

# imagePath = os.path.join("dataset", "without_mask", "20.jpg")


# def detect_mask_image (uploader):
#     confidenceThreshold = 0.5
#     image = cv2.imread(uploader)
#     if image is None:
#         raise FileNotFoundError(f"Could not read input image: {uploader}")
#     orig = image.copy()
#     (h, w) = image.shape[:2]

#     # construct a blob from the image
#     blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300),
#         (104.0, 177.0, 123.0))

#     # pass the blob through the network and obtain the face detections
#     print("[INFO] computing face detections...")
#     net.setInput(blob)
#     detections = net.forward()
    
#     return image, detections, confidenceThreshold

# uploader = widgets.FileUpload(accept="image/*",multiple=False)

# display(uploader)
 
# image, detections, confidenceThreshold = detect_mask_image(uploader)


def detect_mask_image(uploader, confidence_threshold):
    if not uploader.value:
        raise ValueError("Please upload an image before running detection.")

    # Handle ipywidgets 7 and 8
    if isinstance(uploader.value, dict):
        uploaded_file = next(iter(uploader.value.values()))
    else:
        uploaded_file = uploader.value[0]

    image_bytes = uploaded_file["content"]

    if hasattr(image_bytes, "tobytes"):
        image_bytes = image_bytes.tobytes()

    image_array = np.frombuffer(image_bytes, dtype=np.uint8)
    image = cv2.imdecode(image_array, cv2.IMREAD_COLOR)

    if image is None:
        raise ValueError("The uploaded file is not a valid image.")

    orig = image.copy()
    h, w = image.shape[:2]
    confidence_threshold = confidence_threshold

    blob = cv2.dnn.blobFromImage(
        image,
        scalefactor=1.0,
        size=(300, 300),
        mean=(104.0, 177.0, 123.0),
    )

    print("[INFO] computing face detections...")
    net.setInput(blob)
    detections = net.forward()

    print(f"[INFO] loaded image: {w} × {h}")

    return image, orig, detections, confidence_threshold, h, w


uploader = widgets.FileUpload(
    accept="image/*",
    multiple=False,
)

display(uploader)


FileUpload(value=(), accept='image/*', description='Upload')

In [ ]:
image, orig, detections, confidenceThreshold, h, w = detect_mask_image(uploader, 0.20)

# loop over the detections
for i in range(0, detections.shape[2]):
	# extract the confidence (i.e., probability) associated with
	# the detection
	confidence = detections[0, 0, i, 2]

	# filter out weak detections by ensuring the confidence is
	# greater than the minimum confidence
	if confidence > confidenceThreshold:
		# compute the (x, y)-coordinates of the bounding box for
		# the object
		box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
		(startX, startY, endX, endY) = box.astype("int")

		# ensure the bounding boxes fall within the dimensions of
		# the frame
		(startX, startY) = (max(0, startX), max(0, startY))
		(endX, endY) = (min(w - 1, endX), min(h - 1, endY))

		# extract the face ROI, convert it from BGR to RGB channel
		# ordering, resize it to 224x224, and preprocess it
		face = image[startY:endY, startX:endX]
		face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
		face = cv2.resize(face, (224, 224))
		face = img_to_array(face)
		face = preprocess_input(face)
		face = np.expand_dims(face, axis=0)

		# pass the face through the model to determine if the face
		# has a mask or not
		(mask, withoutMask) = model.predict(face)[0]

		# determine the class label and color we'll use to draw
		# the bounding box and text
		label = "Mask" if mask > withoutMask else "No Mask"
		color = (0, 255, 0) if label == "Mask" else (0, 0, 255)

		# include the probability in the label
		label = "{}: {:.2f}%".format(label, max(mask, withoutMask) * 100)

		# display the label and bounding box rectangle on the output
		# frame
		cv2.putText(image, label, (startX, startY - 10),
			cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)
		cv2.rectangle(image, (startX, startY), (endX, endY), color, 2)

# show the output image
cv2.imshow("Output", image)
cv2.waitKey(0)

[INFO] computing face detections...
[INFO] loaded image: 1536 × 1024
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
